# Feature Pipelines
This notebook will compile our previously filtered data, utilizing our data refining pipeline, and feeds the data through a few machine learning models. The goal of this pipeline is reproducability, so we will be configuring the feature pipeline to be exported by the end for future use.

Load the most recent version of our pipeline data: "labeled_step_test," then convert it to Pandas.

In [0]:
from pyspark.sql import SparkSession
import pandas as pd
spark = SparkSession.builder.getOrCreate()
df_spark = spark.table("workspace.silver.labeled_step_test")
df = df_spark.toPandas()
df.head()

,timestamp,sensor_type,distance_cm,device_id,step_label,source_label
0,2025-10-06 20:16:44.778,gyroscope,22,spotter-10.stedi.local,step,device
1,2025-10-06 20:16:44.879,gyroscope,35,spotter-10.stedi.local,step,device
2,2025-10-06 20:16:44.980,gyroscope,108,spotter-10.stedi.local,step,device
3,2025-10-06 20:16:45.081,gyroscope,112,spotter-10.stedi.local,step,device
4,2025-10-06 20:16:45.182,gyroscope,54,spotter-10.stedi.local,step,device


Define which columns we will use as input features; these are the pieces of data we will be measuring for both the pattern recognition and pattern finding.
Numeric
- distance_cm

Categorical
- sensorType
- deviceId

Label
- step_label


In [0]:
feature_cols_numeric = ["distance_cm"]
feature_cols_categorical = ["sensor_type", "device_id"]
label_col = "step_label"

Now we create a train and test split. This way the machine learning system will teach itself and then test that teaching separately, to avoid any improper training and ensure that the ML system functions in the field.

In [0]:
from sklearn.model_selection import train_test_split
X = df[feature_cols_numeric + feature_cols_categorical]
y = df[label_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Up next is building preprocessing steps; both numbers and text requires some processing before we put it into the machine learning system; as the data currently rests it is not readable to the system in any accurate measure.
- Scale the numeric columns

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = StandardScaler()

- One-hot encode categorical columns

In [0]:
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

- Combine into a single transformer

In [0]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, feature_cols_numeric),
        ("cat", categorical_transformer, feature_cols_categorical)
    ]
)

# Build a Scikit-Learn Pipeline
A Scikit-Learn Pipeline chains multiple data transformers (like what we set up previously) and sets it up into a single object. Think of this as applying everything together in a single unified process.

In [0]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline(steps=[
    ("preprocess", preprocessor)
])

# Fit the Pipeline and Transform the Data
Time to pull the ripcord! Fit everything together and set it into a variable to export.

In [0]:
pipeline.fit(X_train)

X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

# Export out the pipeline

In [0]:
import os, joblib

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# get the repo root (everything before /notebooks/)
repo_ws = nb_path.split("/notebooks/")[0]

# local filesystem mirror (writable)
repo_fs = "/Workspace" + repo_ws if not repo_ws.startswith("/Workspace") else repo_ws

out_path = os.path.join(repo_fs, "artifacts", "stedi_feature_pipeline.pkl")
os.makedirs(os.path.dirname(out_path), exist_ok=True)

joblib.dump(pipeline, out_path)
out_path


'/Workspace/Users/hmm733@ensign.edu/csai382_data_training_hmuhlestein/artifacts/stedi_feature_pipeline.pkl'

# Ethics Reflection
I remember learning how to code in python when I was younger and being told the answer code should solve a given list of equations. I slaved over it and worked on it, before ultimately creating a basic calculator, though at the time it felt incredibly complex and worth quite a bit of praise. I remember turning it in and watching my professor use a *different* set of equations, all of which correctly solved through my calculator. When I started to wonder why he would use two different sets of equations, I watched him put in another student's homework, only for it to put out the answers for the initial first set; the student had hard coded the answers in!
I learned that day that if I make something that is both automated and reproducable by anybody, then people can rest assured that any answers that come out of the system are accurate and not tampered with. Much like that first calculator, learning how to correctly create a consistent and reproducible feature pipeline ensures that my answers are truthful and as close to without bias as possible.